In [1]:
# =============================================================================
# GUS02C: Data Unification & Inspection
# =============================================================================
# This notebook demonstrates the v4.2 unification capabilities:
#   1. Load the complete database (from GUS02A/GUS02B)
#   2. Inspect time series (pd.Series with DatetimeIndex)
#   3. Inspect population data (TERYTRecord.pop)
#   4. Inspect urban/rural classification (TERYTRecord.pop_class)
#   5. Inspect category coding (DataSeries.cat_code, cat_bounds)
# =============================================================================

# STEP 1: Imports and Path Setup
import os
import sys
from pathlib import Path
import importlib
import gc
import random

import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

# Find the repository root
def find_repo_root(start=Path.cwd()):
    for p in [start] + list(start.parents):
        if (p / 'Code').exists() or (p / '.git').exists():
            return p
    return start

repo_root = find_repo_root()
tools_path = repo_root / 'Code' / 'tools'
if str(tools_path) not in sys.path:
    sys.path.insert(0, str(tools_path))

import geoTERYT_db as gtdb
importlib.reload(gtdb)

# Paths
data_root = repo_root.parent.parent / 'Data'
geo_root = data_root / 'Geospatial'
gus_root = data_root / 'GUS'

print(f"Repository root: {repo_root}")
print(f"GUS root: {gus_root}")

Repository root: /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/local_repo/LRDWI-Paper
GUS root: /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/GUS


In [ ]:
# =============================================================================
# STEP 2: Load Database
# =============================================================================
complete_db_path = geo_root / 'geoteryt_complete_final.pkl'
db = gtdb.load_complete_database(complete_db_path)
db.print_summary()
print(f"\nData summary: {db.get_data_summary()}")

In [ ]:
# =============================================================================
# STEP 3: Pick a Random TERYTRecord (level=6, with data)
# =============================================================================
# Randomly draw a gmina-level record that has population data

candidates = [r for r in db._records.values() 
              if r.level == 6 and r.has_data and r.pop.notna().any()]
random.seed(42)  # For reproducibility
rec = random.choice(candidates)

print(f"Randomly selected record: {rec.teryt_id} - {rec.name}")
print(f"  Level: {rec.level}, Kind: {rec.kind}, Rodz: {rec.rodz}")
print(f"  Has data: {rec.has_data}, Data series: {rec.n_data_series}")
print(f"  Has cross tables: {rec.has_cross_tables}")
print(f"  Subjects: {rec.list_subjects()}")
print(f"\n--- Full record dict ---")
rec.to_dict()

In [ ]:
# =============================================================================
# STEP 4: Time Series Demo (pd.Series with DatetimeIndex)
# =============================================================================
# Show that data is now stored as pd.Series indexed by datetime

print(f"=== Time Series for {rec.teryt_id} ({rec.name}) ===\n")

# Pick the first DataSeries
if rec.data:
    first_key = list(rec.data.keys())[0]
    ds = rec.data[first_key]
    print(f"DataSeries: {ds}")
    print(f"  Type of values: {type(ds.values)}")
    print(f"  Index type: {type(ds.values.index)}")
    print(f"  Years with data: {ds.years}")
    print(f"\n--- Full pd.Series ---")
    display(ds.values)
    
    # Plot the time series
    fig, ax = plt.subplots(figsize=(12, 4))
    ds.values.dropna().plot(ax=ax, marker='o', linewidth=1.5)
    ax.set_title(f"Time series: {ds.subject_id} / variable {ds.variable_id}\n"
                 f"Categories: {ds.categories}")
    ax.set_xlabel("Year")
    ax.set_ylabel("Value")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# =============================================================================
# STEP 5: Population Data (TERYTRecord.pop)
# =============================================================================

print(f"=== Population for {rec.teryt_id} ({rec.name}) ===\n")
print(f"Type: {type(rec.pop)}")
print(f"Index type: {type(rec.pop.index)}")
print(f"Years with data: {[ts.year for ts in rec.pop.dropna().index]}")
print(f"\n--- Population time series ---")
display(rec.pop)

# Plot population
fig, ax = plt.subplots(figsize=(12, 4))
rec.pop.dropna().plot(ax=ax, marker='o', linewidth=1.5, color='green')
ax.set_title(f"Total population: {rec.teryt_id} ({rec.name})")
ax.set_xlabel("Year")
ax.set_ylabel("Population")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# STEP 6: Urban/Rural Classification (TERYTRecord.pop_class)
# =============================================================================

print(f"=== Pop Classification for {rec.teryt_id} ({rec.name}) ===\n")
print(f"Rodz: {rec.rodz}")
print(f"pop_class type: {type(rec.pop_class)}")
print(f"pop_class shape: {rec.pop_class.shape}")
print(f"\n--- Classification table ---")
display(rec.pop_class)

In [ ]:
# =============================================================================
# STEP 7: Dimension Label Coding (DataSeries.cat_code, cat_bounds)
# =============================================================================

print(f"=== Dimension Coding for {rec.teryt_id} ({rec.name}) ===\n")

# Show a few DataSeries with their cat_codes and cat_bounds
coded_series = [(k, s) for k, s in rec.data.items() if s.cat_code]
print(f"DataSeries with cat_code: {len(coded_series)} / {len(rec.data)}\n")

for key, series in coded_series[:8]:
    print(f"  {series}")
    print(f"    Categories: {series.categories}")
    print(f"    cat_code: {series.cat_code}")
    if series.cat_bounds:
        print(f"    cat_bounds: {series.cat_bounds}")
    print()

# Summary: show unique dimension codings for one subject
if coded_series:
    sample_sid = coded_series[0][1].subject_id
    subj_data = rec.get_data_by_subject(sample_sid)
    print(f"\n=== All coded categories for subject {sample_sid} ===")
    all_codes = {}
    for k, s in subj_data.items():
        for dim, cat in s.categories.items():
            code = s.cat_code.get(dim, '?')
            if dim not in all_codes:
                all_codes[dim] = []
            all_codes[dim].append((cat, code))
    
    for dim, pairs in all_codes.items():
        unique = sorted(set(pairs), key=lambda x: x[1] if isinstance(x[1], (int, float)) else 999)
        print(f"\n  Dimension {dim}:")
        for label, code in unique:
            print(f"    {code}: '{label}'")